[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/061.%20The%20Order%20Batching%20Problem/P61-Tier-3_executed.ipynb)

---


# 61. The Order Batching Problem
## Tier 3: Harmony Search Metaheuristic

### Purpose and scope

This notebook uses Harmony Search to assign customer orders to picker batches. Each batch has a volume limit, and its travel distance is estimated with a nearest-neighbor route that starts and ends at a depot. The algorithm searches for a feasible batching plan with low total estimated travel distance.

The inputs are order volumes and 2D locations, a picker capacity, and Harmony Search parameters. The outputs are the best feasible batching plan found during the search, its estimated distance, progress statistics, baseline comparisons, and visualizations. Harmony Search is a stochastic metaheuristic: it does not certify that the best plan found is globally optimal.

### Model assumptions

- Each order is indivisible and must be assigned to exactly one batch.
- The sum of order volumes in a batch cannot exceed picker capacity.
- The depot is at coordinate $(0,0)$, and travel distance is Euclidean distance in the same coordinate units as the order locations.
- Each batch is routed independently. Its route is estimated by repeatedly visiting the nearest unvisited order, then returning to the depot. This is a route-cost rule used by the evaluation function, not an exact solution of the traveling salesperson problem.
- Picker congestion, aisle restrictions, service times, multiple depots, and interactions between batches are outside this example's scope.

### Batching model and route-cost evaluation

A solution assigns each order to a batch. Equivalently, it partitions the set of orders into nonempty batches. The capacity constraint makes a partition feasible; the objective adds the estimated route cost of every batch:

$$\min_{\mathcal{B}}\quad \sum_{B\in\mathcal{B}} C(B)$$

$$\text{subject to}\quad \sum_{i\in B} v_i\leq Q\quad (B\in\mathcal{B}),\qquad \bigcup_{B\in\mathcal{B}}B=\{0,\ldots,n-1\},\qquad B\cap B'=\varnothing\quad (B\ne B').$$

Here, $n$ is the number of orders, $v_i$ is the volume of order $i$, and $Q$ is picker capacity. The collection $\mathcal{B}$ contains the batches. For a batch $B$, $C(B)$ is the distance of the nearest-neighbor tour, including the return to the depot. A one-order batch therefore has a positive round-trip cost unless its location is the depot. In the code, `solution` stores the assignment, `evaluate_fitness` checks feasibility and sums costs, and `calculate_batch_cost` applies the route rule.

### Harmony Search: intuition and formal outline

Harmony Search builds a memory of good feasible solutions, then repeatedly creates a candidate solution by borrowing assignments from that memory, adjusting some assignments, or choosing assignments at random. The musical analogy is useful for remembering the roles of memory and adjustment; the algorithm itself is a stochastic search over batch assignments.

Let $H$ be the harmony memory, containing at most `HMS` feasible assignment vectors. In one improvisation, each order is considered in turn. With probability `HMCR`, the algorithm selects that order's batch from a randomly chosen harmony in $H$; otherwise it makes a random feasible assignment. A memory-based assignment may be pitch-adjusted with probability `PAR`: the implementation considers feasible neighboring batches from the selected harmony and chooses the one with the most similar mean order volume. Capacity checks are applied while constructing the candidate, so every completed candidate remains feasible.

After evaluating a candidate, the algorithm replaces the worst harmony only if the candidate has lower cost. Thus the best fitness in memory cannot increase. At specified iteration intervals, `PAR` is reduced slightly. The initial memory, improvisation, evaluation, memory update, and progress tracking are all visible in `HarmonySearchOptimizer`.

In standard Harmony Search terminology, `HMCR` is the Harmony Memory Considering Rate and `PAR` is the Pitch Adjusting Rate. `HMS` is the Harmony Memory Size, and `NI` is the Number of Improvisations. A lower fitness means a shorter estimated total route distance. The memory's diversity statistic in this notebook is the standard deviation of fitness values; it is not a direct measure of how different the batch assignments are.

For $n$ orders, evaluating the nearest-neighbor route costs at most $O(n^2)$ distance comparisons across all batches. Pitch-adjustment similarity calculations can take up to $O(n^3)$ work in one improvisation in this implementation. Sorting the memory costs $O(\text{HMS}\log(\text{HMS}))$ when an improving candidate is inserted. These are worst-case bounds; practical work depends on the batch structure and how often pitch adjustment occurs. Memory storage is $O(\text{HMS}\,n)$.

### What to examine

- The convergence plot shows the best and average fitness values in memory as improvisations proceed.
- The diversity plot shows the spread of memory fitness values, not assignment-level diversity.
- Baseline and parameter-sensitivity results are outcomes of stochastic runs, not guarantees about all instances or parameter settings.
- A low fitness is meaningful only together with feasibility: every order must be assigned once and every batch must respect capacity.

In [ ]:
# Import the libraries used for the search, analysis, and plots.
import math
import random
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Fix both random-number generators so that a run can be reproduced.
np.random.seed(42)
random.seed(42)

print("Order Batching with Harmony Search")
print("Required packages imported successfully.")

In [ ]:
class Order:
    """Represent a customer order by its volume and 2D location."""

    def __init__(self, id, volume, location):
        self.id = id
        self.volume = volume
        self.location = location

    def __repr__(self):
        return f"Order({self.id}, volume={self.volume}, location={self.location})"


class HarmonySearchOptimizer:
    """Apply Harmony Search to capacity-constrained order batching."""

    def __init__(self, orders, capacity, HMS=20, HMCR=0.85, PAR=0.3, NI=5000):
        if not orders:
            raise ValueError("At least one order is required.")
        if capacity <= 0:
            raise ValueError("Capacity must be positive.")
        if any(order.volume <= 0 or order.volume > capacity for order in orders):
            raise ValueError("Every order volume must be positive and no greater than capacity.")
        if HMS < 1:
            raise ValueError("HMS must be at least 1.")
        if not 0 <= HMCR <= 1 or not 0 <= PAR <= 1:
            raise ValueError("HMCR and PAR must be between 0 and 1.")
        if NI < 1:
            raise ValueError("NI must be at least 1.")

        self.orders = orders
        self.capacity = capacity
        self.num_orders = len(orders)

        # Keep the requested settings so repeated calls start consistently.
        self.HMS = HMS
        self.HMCR = HMCR
        self.initial_PAR = PAR
        self.PAR = PAR
        self.NI = NI

        self.harmony_memory = []
        self.iteration_history = []
        self.best_fitness_history = []
        self.avg_fitness_history = []
        self.diversity_history = []
        self.distances = self._calculate_distances()

    def _calculate_distances(self):
        """Calculate Euclidean distances between order locations."""
        distances = np.zeros((self.num_orders, self.num_orders))
        for i in range(self.num_orders):
            for j in range(self.num_orders):
                loc1 = self.orders[i].location
                loc2 = self.orders[j].location
                distances[i, j] = math.hypot(loc1[0] - loc2[0], loc1[1] - loc2[1])
        return distances

    def generate_random_solution(self):
        """Create a random assignment while respecting batch capacity."""
        solution = {}
        batch_loads = defaultdict(int)
        order_indices = list(range(self.num_orders))
        random.shuffle(order_indices)

        for order_id in order_indices:
            order_volume = self.orders[order_id].volume
            valid_batches = [
                batch_id
                for batch_id, load in batch_loads.items()
                if load + order_volume <= self.capacity
            ]
            new_batch = max(batch_loads.keys()) + 1 if batch_loads else 0
            valid_batches.append(new_batch)

            chosen_batch = random.choice(valid_batches)
            solution[order_id] = chosen_batch
            batch_loads[chosen_batch] += order_volume

        return solution

    def calculate_batch_cost(self, batch_orders):
        """Estimate a closed batch tour using the nearest-neighbor rule."""
        if not batch_orders:
            return 0.0

        depot = (0, 0)
        locations = [self.orders[order_id].location for order_id in batch_orders]
        unvisited = set(range(len(locations)))
        current_location = depot
        total_distance = 0.0

        while unvisited:
            # Breaking distance ties by order index makes route construction deterministic.
            nearest = min(
                unvisited,
                key=lambda index: (
                    math.hypot(
                        current_location[0] - locations[index][0],
                        current_location[1] - locations[index][1],
                    ),
                    index,
                ),
            )
            next_location = locations[nearest]
            total_distance += math.hypot(
                current_location[0] - next_location[0],
                current_location[1] - next_location[1],
            )
            current_location = next_location
            unvisited.remove(nearest)

        # Every picker tour returns to the depot, including a one-order tour.
        total_distance += math.hypot(
            current_location[0] - depot[0],
            current_location[1] - depot[1],
        )
        return total_distance

    def evaluate_fitness(self, solution):
        """Return total estimated distance, rejecting infeasible assignments."""
        if set(solution) != set(range(self.num_orders)):
            raise ValueError("A solution must assign every order exactly once.")

        batches = defaultdict(list)
        for order_id, batch_id in solution.items():
            batches[batch_id].append(order_id)

        total_distance = 0.0
        for batch_orders in batches.values():
            batch_volume = sum(self.orders[i].volume for i in batch_orders)
            if batch_volume > self.capacity:
                raise ValueError("A solution contains a batch that exceeds capacity.")
            total_distance += self.calculate_batch_cost(batch_orders)

        return total_distance

    def initialize_harmony_memory(self):
        """Initialize memory with randomly generated feasible solutions."""
        print(f"Initializing harmony memory with {self.HMS} random solutions...")
        self.harmony_memory = []

        for _ in range(self.HMS):
            solution = self.generate_random_solution()
            fitness = self.evaluate_fitness(solution)
            self.harmony_memory.append((solution, fitness))

        self.harmony_memory.sort(key=lambda item: item[1])
        fitness_values = [fitness for _, fitness in self.harmony_memory]
        print("Initial harmony memory created.")
        print(f"Best initial fitness: {fitness_values[0]:.2f}")
        print(f"Average initial fitness: {np.mean(fitness_values):.2f}")
        print(f"Fitness range: [{min(fitness_values):.2f}, {max(fitness_values):.2f}]")

    def get_neighboring_batches(self, current_batch, harmony_solution):
        """List other batch labels present in a harmony."""
        return [
            batch_id
            for batch_id in set(harmony_solution.values())
            if batch_id != current_batch
        ]

    def get_valid_batches(self, order_id, current_harmony, capacity):
        """Find existing or new batches that can fit the next order."""
        order_volume = self.orders[order_id].volume
        batch_loads = defaultdict(int)

        # Only already assigned orders contribute to the partial candidate's loads.
        for assigned_order_id, batch_id in current_harmony:
            if assigned_order_id != order_id:
                batch_loads[batch_id] += self.orders[assigned_order_id].volume

        valid_batches = [
            batch_id
            for batch_id, load in batch_loads.items()
            if load + order_volume <= capacity
        ]
        new_batch = max(batch_loads.keys()) + 1 if batch_loads else 0
        valid_batches.append(new_batch)
        return valid_batches

    def calculate_batch_similarity(self, batch1, batch2, harmony_solution):
        """Compare batches by their mean order volumes."""
        orders1 = [order_id for order_id, batch_id in harmony_solution.items() if batch_id == batch1]
        orders2 = [order_id for order_id, batch_id in harmony_solution.items() if batch_id == batch2]

        if not orders1 or not orders2:
            return 0.0

        mean1 = np.mean([self.orders[order_id].volume for order_id in orders1])
        mean2 = np.mean([self.orders[order_id].volume for order_id in orders2])
        return 1.0 - abs(mean1 - mean2) / max(mean1, mean2)

    def improvise_new_harmony(self):
        """Construct one feasible candidate using memory, adjustment, and randomness."""
        new_harmony = []

        for order_id in range(self.num_orders):
            valid_batches = self.get_valid_batches(
                order_id, new_harmony, self.capacity
            )

            if random.random() < self.HMCR:
                selected_harmony, _ = random.choice(self.harmony_memory)
                proposed_batch = selected_harmony[order_id]

                # A memory assignment is usable only when it fits the partial candidate.
                batch = (
                    proposed_batch
                    if proposed_batch in valid_batches
                    else random.choice(valid_batches)
                )

                if batch == proposed_batch and random.random() < self.PAR:
                    neighbors = self.get_neighboring_batches(
                        proposed_batch, selected_harmony
                    )
                    feasible_neighbors = [
                        neighbor
                        for neighbor in neighbors
                        if neighbor in valid_batches
                    ]
                    if feasible_neighbors:
                        batch = max(
                            feasible_neighbors,
                            key=lambda neighbor: self.calculate_batch_similarity(
                                neighbor, proposed_batch, selected_harmony
                            ),
                        )
            else:
                batch = random.choice(valid_batches)

            new_harmony.append((order_id, batch))

        return {order_id: batch_id for order_id, batch_id in new_harmony}

    def update_harmony_memory(self, new_solution, new_fitness):
        """Replace the worst stored harmony when the candidate is better."""
        if new_fitness < self.harmony_memory[-1][1]:
            self.harmony_memory[-1] = (new_solution, new_fitness)
            self.harmony_memory.sort(key=lambda item: item[1])
            return True
        return False

    def calculate_diversity(self):
        """Measure the spread of fitness values in the harmony memory."""
        if len(self.harmony_memory) < 2:
            return 0.0
        fitness_values = [fitness for _, fitness in self.harmony_memory]
        return float(np.std(fitness_values))

    def optimize(self):
        """Run the Harmony Search iterations and return the best stored plan."""
        self.PAR = self.initial_PAR
        self.iteration_history = []
        self.best_fitness_history = []
        self.avg_fitness_history = []
        self.diversity_history = []

        print("\nStarting Harmony Search optimization.")
        print(
            f"Parameters: HMS={self.HMS}, HMCR={self.HMCR}, "
            f"PAR={self.PAR}, NI={self.NI}"
        )
        start_time = time.time()
        self.initialize_harmony_memory()

        for iteration in range(self.NI):
            new_solution = self.improvise_new_harmony()
            new_fitness = self.evaluate_fitness(new_solution)
            self.update_harmony_memory(new_solution, new_fitness)

            best_fitness = self.harmony_memory[0][1]
            avg_fitness = np.mean(
                [fitness for _, fitness in self.harmony_memory]
            )
            diversity = self.calculate_diversity()

            self.iteration_history.append(iteration)
            self.best_fitness_history.append(best_fitness)
            self.avg_fitness_history.append(avg_fitness)
            self.diversity_history.append(diversity)

            if iteration > 0 and iteration % 1000 == 0:
                self.PAR *= 0.99

            if iteration % 500 == 0:
                print(
                    f"Iteration {iteration:4d}: Best={best_fitness:8.2f}, "
                    f"Average={avg_fitness:8.2f}, Diversity={diversity:6.2f}"
                )

        optimization_time = time.time() - start_time
        best_solution, best_fitness = self.harmony_memory[0]
        results = {
            "best_solution": best_solution,
            "best_fitness": best_fitness,
            "optimization_time": optimization_time,
            "iterations": self.NI,
            "initial_PAR": self.initial_PAR,
            "final_PAR": self.PAR,
            "convergence_data": {
                "iterations": self.iteration_history,
                "best_fitness": self.best_fitness_history,
                "avg_fitness": self.avg_fitness_history,
                "diversity": self.diversity_history,
            },
        }

        print("\nOptimization completed.")
        print(f"Best fitness found: {best_fitness:.2f}")
        print(f"Optimization time: {optimization_time:.2f} seconds")
        return results


print("Harmony Search optimizer defined successfully.")

In [ ]:
# Define the example instance: 12 orders, total volume 30, and capacity 8.
orders_example = [
    Order(0, 3, (2, 8)),
    Order(1, 2, (5, 3)),
    Order(2, 4, (8, 12)),
    Order(3, 1, (3, 6)),
    Order(4, 3, (9, 4)),
    Order(5, 2, (6, 10)),
    Order(6, 3, (4, 7)),
    Order(7, 2, (7, 9)),
    Order(8, 1, (5, 11)),
    Order(9, 4, (10, 6)),
    Order(10, 2, (3, 4)),
    Order(11, 3, (8, 5)),
]

# These are the settings used for the main Harmony Search run.
optimizer = HarmonySearchOptimizer(
    orders=orders_example,
    capacity=8,
    HMS=15,
    HMCR=0.9,
    PAR=0.25,
    NI=3200,
)

total_volume = sum(order.volume for order in orders_example)
minimum_batches_by_volume = math.ceil(total_volume / optimizer.capacity)

print("Problem instance")
print(f"Number of orders: {len(orders_example)}")
print(f"Total volume: {total_volume}")
print(f"Picker capacity: {optimizer.capacity}")
print(f"Volume-based lower bound on batch count: {minimum_batches_by_volume}")

print("\nOrder details:")
for order in orders_example:
    print(
        f"  Order {order.id}: volume={order.volume}, "
        f"location={order.location}"
    )

print("\nHarmony Search parameters:")
print(f"  HMS (Harmony Memory Size): {optimizer.HMS}")
print(f"  HMCR (Harmony Memory Considering Rate): {optimizer.HMCR}")
print(f"  PAR (initial Pitch Adjusting Rate): {optimizer.initial_PAR}")
print(f"  NI (Number of Improvisations): {optimizer.NI}")

In [ ]:
# Run the search, then inspect the best feasible plan stored in memory.
results = optimizer.optimize()
best_solution = results["best_solution"]
best_fitness = results["best_fitness"]

batches = defaultdict(list)
for order_id, batch_id in best_solution.items():
    batches[batch_id].append(order_id)

# These assertions link the reported plan to the model's assignment and capacity rules.
assert set(best_solution) == set(range(len(orders_example)))
for order_list in batches.values():
    assert sum(orders_example[i].volume for i in order_list) <= optimizer.capacity
assert math.isclose(
    sum(optimizer.calculate_batch_cost(order_list) for order_list in batches.values()),
    best_fitness,
    rel_tol=1e-9,
    abs_tol=1e-9,
)

print("\nBest solution analysis")
print(f"Best estimated total distance: {best_fitness:.2f}")
for batch_id, order_list in sorted(batches.items()):
    batch_volume = sum(optimizer.orders[i].volume for i in order_list)
    batch_cost = optimizer.calculate_batch_cost(order_list)
    print(
        f"  Batch {batch_id}: orders {order_list}, "
        f"volume {batch_volume}/{optimizer.capacity}, "
        f"estimated distance {batch_cost:.2f}"
    )

total_picked_volume = sum(order.volume for order in optimizer.orders)
average_utilization = (
    total_picked_volume / (len(batches) * optimizer.capacity) * 100
)
print("\nPerformance measures")
print(f"Number of batches: {len(batches)}")
print(f"Average batch utilization: {average_utilization:.1f}%")
print(f"Optimization time: {results['optimization_time']:.2f} seconds")
print(f"Improvisations completed: {len(results['convergence_data']['iterations'])}")

In [ ]:
# Compare the search with a first-fit heuristic and a random-solution baseline.
def greedy_first_fit(orders, capacity):
    """Assign orders in input order to the first batch with sufficient capacity."""
    solution = {}
    batch_loads = defaultdict(int)

    for order_id, order in enumerate(orders):
        assigned = False
        for batch_id in sorted(batch_loads):
            if batch_loads[batch_id] + order.volume <= capacity:
                solution[order_id] = batch_id
                batch_loads[batch_id] += order.volume
                assigned = True
                break

        if not assigned:
            new_batch = max(batch_loads) + 1 if batch_loads else 0
            solution[order_id] = new_batch
            batch_loads[new_batch] += order.volume

    return solution


def random_baseline(orders, capacity, trials=100):
    """Return the best of a specified number of random feasible solutions."""
    if trials < 1:
        raise ValueError("The number of trials must be at least 1.")

    temp_optimizer = HarmonySearchOptimizer(orders, capacity)
    best_solution = None
    best_fitness = float("inf")

    for _ in range(trials):
        solution = temp_optimizer.generate_random_solution()
        fitness = temp_optimizer.evaluate_fitness(solution)
        if fitness < best_fitness:
            best_solution = solution
            best_fitness = fitness

    return best_solution, best_fitness


greedy_solution = greedy_first_fit(orders_example, optimizer.capacity)
greedy_fitness = optimizer.evaluate_fitness(greedy_solution)
random_solution, random_fitness = random_baseline(
    orders_example, optimizer.capacity, trials=100
)

print("Baseline comparisons")
print(f"Greedy first-fit: {greedy_fitness:.2f}")
print(f"Best of 100 random solutions: {random_fitness:.2f}")
print(f"Harmony Search best found: {best_fitness:.2f}")

def percentage_improvement(reference, candidate):
    """Calculate relative distance reduction when the reference is positive."""
    if math.isclose(reference, 0.0, abs_tol=1e-12):
        return 0.0 if math.isclose(candidate, 0.0, abs_tol=1e-12) else float("nan")
    return (reference - candidate) / reference * 100


improvement_vs_random = percentage_improvement(random_fitness, best_fitness)
improvement_vs_greedy = percentage_improvement(greedy_fitness, best_fitness)
print("\nRelative distance reduction (positive means lower distance):")
print(f"  Versus the random baseline: {improvement_vs_random:.1f}%")
print(f"  Versus greedy first-fit: {improvement_vs_greedy:.1f}%")

print("\nGreedy first-fit batch plan:")
greedy_batches = defaultdict(list)
for order_id, batch_id in greedy_solution.items():
    greedy_batches[batch_id].append(order_id)

for batch_id, order_list in sorted(greedy_batches.items()):
    batch_volume = sum(optimizer.orders[i].volume for i in order_list)
    batch_cost = optimizer.calculate_batch_cost(order_list)
    print(
        f"  Batch {batch_id}: orders {order_list}, "
        f"volume {batch_volume}/{optimizer.capacity}, "
        f"estimated distance {batch_cost:.2f}"
    )

In [ ]:
# Inspect recorded memory statistics at evenly spaced points in the run.
def analyze_harmony_memory_evolution(optimizer, random_reference):
    """Report best fitness, mean fitness, and fitness spread at selected iterations."""
    history_length = len(optimizer.iteration_history)
    if history_length == 0:
        raise ValueError("Run the optimizer before analyzing its history.")

    stage_indices = [
        0,
        history_length // 4,
        history_length // 2,
        3 * history_length // 4,
        history_length - 1,
    ]
    stage_names = ["Start", "25%", "50%", "75%", "Final"]

    print("Harmony memory evolution")
    for stage_index, stage_name in zip(stage_indices, stage_names):
        best_at_stage = optimizer.best_fitness_history[stage_index]
        average_at_stage = optimizer.avg_fitness_history[stage_index]
        diversity_at_stage = optimizer.diversity_history[stage_index]
        improvement = percentage_improvement(random_reference, best_at_stage)

        print(
            f"\n{stage_name} stage (recorded iteration "
            f"{optimizer.iteration_history[stage_index]}):"
        )
        print(f"  Best fitness: {best_at_stage:.2f}")
        print(f"  Average fitness: {average_at_stage:.2f}")
        print(f"  Fitness standard deviation: {diversity_at_stage:.2f}")
        print(f"  Distance reduction versus random baseline: {improvement:.1f}%")


analyze_harmony_memory_evolution(optimizer, random_fitness)

In [ ]:
# Compare selected Harmony Search settings on the same order instance.
def parameter_sensitivity_analysis():
    """Run the stated parameter configurations and return their measurements."""
    parameter_sets = [
        {"HMS": 10, "HMCR": 0.8, "PAR": 0.2, "name": "Conservative"},
        {"HMS": 15, "HMCR": 0.9, "PAR": 0.25, "name": "Balanced (Default)"},
        {"HMS": 20, "HMCR": 0.95, "PAR": 0.3, "name": "Aggressive"},
        {"HMS": 12, "HMCR": 0.85, "PAR": 0.15, "name": "Low PAR"},
    ]
    records = []

    print("Parameter sensitivity analysis")
    for parameters in parameter_sets:
        print(f"\nTesting {parameters['name']} configuration...")
        test_optimizer = HarmonySearchOptimizer(
            orders=orders_example,
            capacity=optimizer.capacity,
            HMS=parameters["HMS"],
            HMCR=parameters["HMCR"],
            PAR=parameters["PAR"],
            NI=2000,
        )
        test_results = test_optimizer.optimize()
        records.append(
            {
                "name": parameters["name"],
                "HMS": parameters["HMS"],
                "HMCR": parameters["HMCR"],
                "PAR": parameters["PAR"],
                "best_fitness": test_results["best_fitness"],
                "time_seconds": test_results["optimization_time"],
            }
        )
        print(f"  Best fitness found: {test_results['best_fitness']:.2f}")
        print(f"  Run time: {test_results['optimization_time']:.2f} seconds")

    return pd.DataFrame(records)


sensitivity_df = parameter_sensitivity_analysis()
print("\nParameter sensitivity results:")
print(sensitivity_df.round(2).to_string(index=False))

In [ ]:
# Plot convergence, memory fitness spread, baseline results, and parameter runs.
def create_visualizations(results, sensitivity_df):
    """Create four plots that summarize the recorded search results."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Order Batching: Harmony Search Results",
        fontsize=16,
        fontweight="bold",
    )

    convergence = results["convergence_data"]

    # Best and mean fitness show how the stored population changes over iterations.
    ax1 = axes[0, 0]
    ax1.plot(
        convergence["iterations"],
        convergence["best_fitness"],
        label="Best fitness",
        linewidth=2,
    )
    ax1.plot(
        convergence["iterations"],
        convergence["avg_fitness"],
        label="Average fitness",
        linestyle="--",
        alpha=0.8,
    )
    ax1.set_xlabel("Improvisation iteration")
    ax1.set_ylabel("Estimated total distance (coordinate units)")
    ax1.set_title("Convergence")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # This spread is based on fitness values, not on assignment differences.
    ax2 = axes[0, 1]
    ax2.plot(
        convergence["iterations"],
        convergence["diversity"],
        color="purple",
        linewidth=2,
    )
    ax2.set_xlabel("Improvisation iteration")
    ax2.set_ylabel("Fitness standard deviation")
    ax2.set_title("Harmony Memory Fitness Spread")
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    method_names = ["Best of 100 random", "Greedy first-fit", "Harmony Search"]
    method_fitness = [random_fitness, greedy_fitness, results["best_fitness"]]
    bars = ax3.bar(method_names, method_fitness, color=["#e07a5f", "#4f81bd", "#66a061"])
    ax3.set_ylabel("Estimated total distance (coordinate units)")
    ax3.set_title("Baseline Comparison")
    ax3.tick_params(axis="x", labelrotation=15)
    for bar, value in zip(bars, method_fitness):
        ax3.annotate(
            f"{value:.1f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
        )

    ax4 = axes[1, 1]
    positions = np.arange(len(sensitivity_df))
    bars = ax4.bar(
        positions,
        sensitivity_df["best_fitness"],
        color="skyblue",
    )
    ax4.set_xlabel("Parameter configuration")
    ax4.set_ylabel("Best estimated total distance")
    ax4.set_title("Parameter Sensitivity Runs")
    ax4.set_xticks(positions)
    ax4.set_xticklabels(sensitivity_df["name"], rotation=30, ha="right")
    for bar, value in zip(bars, sensitivity_df["best_fitness"]):
        ax4.annotate(
            f"{value:.1f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    fig.tight_layout()
    plt.show()
    return fig


visualization_fig = create_visualizations(results, sensitivity_df)
print("Visualizations created successfully.")

In [ ]:
# Relate the algorithm's parameters and recorded measures to the search process.
print("Harmony Search algorithm analysis")

print("\nMusical improvisation analogy:")
print("  Harmony memory represents previously found feasible assignments.")
print("  HMCR controls how often assignments are drawn from that memory.")
print("  PAR controls how often a memory-based assignment is adjusted.")
print("  Random assignment contributes exploration of other feasible plans.")

print("\nComputational complexity:")
print("  Route evaluation is O(n^2) in the worst case for n orders.")
print("  Pitch-adjustment similarity work can be O(n^3) per improvisation.")
print("  A memory sort costs O(HMS log(HMS)) after a candidate improves the worst entry.")
print("  Stored harmonies require O(HMS * n) space.")
print(f"  This run used NI={optimizer.NI}, HMS={optimizer.HMS}, and n={len(orders_example)}.")
print(f"  Measured optimization time: {results['optimization_time']:.2f} seconds.")

print("\nParameter roles:")
print(f"  HMS (memory size): {optimizer.HMS}")
print("    A larger memory stores more candidate assignments, with additional evaluation and storage work.")
print(f"  HMCR (memory consideration rate): {optimizer.HMCR}")
print("    A higher HMCR favors assignments from stored harmonies; random feasible assignments occur less often.")
print(f"  PAR (initial pitch-adjusting rate): {results['initial_PAR']}")
print(f"  PAR after the scheduled reductions: {results['final_PAR']:.4f}")
print("    Pitch adjustment considers feasible neighboring batches from a selected harmony.")

print("\nObserved run measures:")
print(f"  Best estimated distance found: {best_fitness:.2f}")
print(f"  Recorded improvisations: {len(results['convergence_data']['iterations'])}")
print(f"  Final memory fitness spread: {results['convergence_data']['diversity'][-1]:.2f}")
print(f"  Distance reduction versus best random baseline: {improvement_vs_random:.1f}%")
print(f"  Distance reduction versus greedy first-fit: {improvement_vs_greedy:.1f}%")

print("\nInterpretation and limitations:")
print("  Harmony Search is stochastic, so different seeds can produce different plans.")
print("  The best stored fitness is non-increasing, but a low value is not a proof of global optimality.")
print("  The route objective uses a nearest-neighbor estimate and inherits that rule's limitations.")
print("  Parameter sensitivity should be interpreted across repeated runs when assessing reliability.")

In [ ]:
# Summarize the measurements generated by this run without assuming a particular outcome.
print("Summary and conclusions")
print("\nRun results:")
print(f"  Best estimated total distance found: {best_fitness:.2f}")
print(f"  Distance reduction versus best of 100 random plans: {improvement_vs_random:.1f}%")
print(f"  Distance reduction versus greedy first-fit: {improvement_vs_greedy:.1f}%")
print(f"  Improvisations recorded: {len(results['convergence_data']['iterations'])}")
print(f"  Measured optimization time: {results['optimization_time']:.2f} seconds")
print(f"  Final fitness standard deviation in memory: {results['convergence_data']['diversity'][-1]:.2f}")

print("\nMethod insights:")
print("  Harmony Search combines memory-based assignments, pitch adjustment, and random feasible assignments.")
print("  Capacity-aware construction keeps candidate batch plans feasible.")
print("  Replacing the worst harmony only with a better candidate preserves the best stored fitness.")
print("  The search result is a best-found plan, not an optimality certificate.")

print("\nScope of the findings:")
print("  The numerical objective is total nearest-neighbor distance for the stated order locations and depot.")
print("  The random baseline reports the best solution among its specified trials, not the trial average.")
print("  Parameter comparisons are stochastic experiments on this instance and do not establish universal rankings.")
print("  Larger instances, additional operational constraints, and repeated-seed studies are natural extensions.")